In [1]:
"""
Build daily temperature statistics per station for mainland France.

Output CSV columns:
    date, station_id, region_code, tmin, tmax, tavg, tmean_calc

- Uses station_to_reg.csv to know which stations to pull
- Filters out overseas regions (region_code starting with 97 or 98)
- Fetches daily data from Meteostat (tmin, tmax, tavg when available)
"""

from datetime import date
from pathlib import Path

import pandas as pd
from meteostat import daily


# =============================================================================
# CONFIG
# =============================================================================

START_DATE = date(2000, 1, 1)
END_DATE   = date(2024, 12, 31)

# You are running from HBA/, mapping & output live in ../EDA/
STATION_TO_REG_PATH = Path("../EDA/station_to_reg.csv")
OUTPUT_CSV_PATH     = Path("../EDA/stations_daily_statistics_mainland.csv")


# =============================================================================
# HELPERS
# =============================================================================

def load_station_to_region(path: Path) -> pd.DataFrame:
    """
    Load station→region mapping and filter to mainland France.

    Expected columns in CSV:
        station_id, region_code
    """
    print(f"Reading station_to_reg from: {path.resolve()}")
    df = pd.read_csv(path)

    # Basic check
    expected_cols = {"station_id", "region_code"}
    if not expected_cols.issubset(df.columns):
        raise ValueError(
            f"station_to_reg.csv must contain columns {expected_cols}, "
            f"but has {list(df.columns)}"
        )

    # Ensure string types
    df["station_id"] = df["station_id"].astype(str)
    df["region_code"] = df["region_code"].astype(str)

    # Filter to mainland: exclude overseas regions (97*, 98*)
    mask_overseas = df["region_code"].str.startswith(("97", "98"))
    df_mainland = df[~mask_overseas].copy()

    if df_mainland.empty:
        raise ValueError("After filtering, no mainland stations remain. Check station_to_reg.csv.")

    return df_mainland


def fetch_station_daily_statistics(
    station_ids,
    station_to_reg: pd.DataFrame,
    start: date,
    end: date,
) -> pd.DataFrame:
    """
    Fetch daily min, max, avg temperatures per station from Meteostat.

    Returns a DataFrame with columns:
        date, station_id, region_code, tmin, tmax, tavg, tmean_calc
    """
    all_frames = []

    for sid in station_ids:
        print(f"Fetching Meteostat daily data for station {sid}...")
        ts = daily(sid, start, end)   # TimeSeries
        df = ts.fetch()               # DataFrame

        if df is None or df.empty:
            print(f"  Warning: no data for station {sid} in given period.")
            continue

        # Collect available temperature columns
        cols = [c for c in ["tmin", "tmax", "tavg"] if c in df.columns]
        if not cols:
            print(f"  Warning: no temperature columns for station {sid}, skipping.")
            continue

        out = df[cols].reset_index()  # index is 'time'
        out.rename(columns={"time": "date"}, inplace=True)
        out["station_id"] = str(sid)

        all_frames.append(out)

    if not all_frames:
        raise ValueError("No station data fetched from Meteostat. Check IDs, date range, or connectivity.")

    station_df = pd.concat(all_frames, ignore_index=True)

    # Ensure types
    station_df["date"] = pd.to_datetime(station_df["date"])
    station_df["station_id"] = station_df["station_id"].astype(str)

    # Merge region_code onto each station row
    station_df = station_df.merge(
        station_to_reg,
        on="station_id",
        how="left",
    )

    if station_df["region_code"].isna().any():
        missing = station_df[station_df["region_code"].isna()]["station_id"].unique()
        print(f"Warning: {len(missing)} stations have no region_code in mapping: {missing}")

    # Compute mean from min & max if both available
    if "tmin" in station_df.columns and "tmax" in station_df.columns:
        station_df["tmean_calc"] = (station_df["tmin"] + station_df["tmax"]) / 2.0
    else:
        station_df["tmean_calc"] = pd.NA

    # Order columns nicely
    final_cols = ["date", "station_id", "region_code"]
    for col in ["tmin", "tmax", "tavg", "tmean_calc"]:
        if col in station_df.columns:
            final_cols.append(col)

    station_df = station_df[final_cols]

    return station_df


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("Loading station_to_reg mapping...")
    station_to_reg = load_station_to_region(STATION_TO_REG_PATH)

    n_stations = station_to_reg["station_id"].nunique()
    n_regions = station_to_reg["region_code"].nunique()
    print(f"Found {n_stations} mainland stations across {n_regions} regions.")

    station_ids = sorted(station_to_reg["station_id"].unique().tolist())
    print(f"Fetching daily data for {len(station_ids)} stations from {START_DATE} to {END_DATE}...")

    station_daily_stats = fetch_station_daily_statistics(
        station_ids=station_ids,
        station_to_reg=station_to_reg,
        start=START_DATE,
        end=END_DATE,
    )

    print(f"Downloaded {len(station_daily_stats):,} rows of station daily statistics.")
    print(f"Saving to {OUTPUT_CSV_PATH.resolve()} ...")

    OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    station_daily_stats.to_csv(OUTPUT_CSV_PATH, index=False)

    # Small summary
    date_min = station_daily_stats["date"].min()
    date_max = station_daily_stats["date"].max()
    print(f"Done. Date range in output: {date_min.date()} → {date_max.date()}")


if __name__ == "__main__":
    main()

Loading station_to_reg mapping...
Reading station_to_reg from: /Users/marielouiselysholt/Desktop/Master/Masters_2026/EDA/station_to_reg.csv
Found 197 mainland stations across 13 regions.
Fetching daily data for 197 stations from 2000-01-01 to 2024-12-31...
Fetching Meteostat daily data for station 07002...
Fetching Meteostat daily data for station 07003...
Fetching Meteostat daily data for station 07005...
Fetching Meteostat daily data for station 07010...
Fetching Meteostat daily data for station 07015...
Fetching Meteostat daily data for station 07017...
Fetching Meteostat daily data for station 07020...
Fetching Meteostat daily data for station 07022...
Fetching Meteostat daily data for station 07024...
Fetching Meteostat daily data for station 07027...
Fetching Meteostat daily data for station 07029...
Fetching Meteostat daily data for station 07031...
Fetching Meteostat daily data for station 07033...
Fetching Meteostat daily data for station 07037...
Fetching Meteostat daily data